# Phase 3 — RAG System Exploration

This notebook demonstrates the Banking AI Copilot RAG pipeline:
1. Document loading and chunking
2. Embedding and vector store indexing
3. Similarity search with metadata filtering
4. Full QA chain with source citations
5. Customer context retrieval

**Run from the project root:** `jupyter notebook notebooks/01_rag_exploration.ipynb`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

os.environ.setdefault('DATABASE_URL', 'sqlite:///../banking.db')
os.environ.setdefault('EMBEDDING_BACKEND', 'sentence_transformers')
os.environ.setdefault('VECTOR_DB', 'chroma')
os.environ.setdefault('CHROMA_PERSIST_DIR', '../data/chroma_db')
os.environ.setdefault('CHROMA_COLLECTION', 'banking_docs')

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment configured.')

## 1. Document Loading

In [ ]:
from src.rag.document_loader import load_banking_documents

docs = load_banking_documents('../data/raw/banking_docs')
print(f'Loaded {len(docs)} raw documents')
print('\nSample document:')
print(f'  Source: {docs[0].metadata["source"]}')
print(f'  Content preview: {docs[0].page_content[:200]}...')

## 2. Text Chunking

In [ ]:
from src.rag.chunker import chunk_documents

chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)
print(f'Created {len(chunks)} chunks from {len(docs)} documents')
print(f'Average chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars')
print('\nSample chunk metadata:')
print(chunks[0].metadata)

## 3. Vector Store — Similarity Search

In [ ]:
from src.rag.vector_store import similarity_search

queries = [
    'What is the AML reporting threshold for cash transactions?',
    'What credit score is needed for a premium credit card?',
    'What are the wire transfer fees for international transfers?',
]

for q in queries:
    print(f'\nQuery: {q}')
    results = similarity_search(q, k=3)
    for i, doc in enumerate(results, 1):
        print(f'  [{i}] {doc.metadata.get("source", "?")} — {doc.page_content[:120]}...')

## 4. Full QA Chain with Citations

In [ ]:
if not os.getenv('OPENAI_API_KEY'):
    print('OPENAI_API_KEY not set — skipping QA chain demo (retrieval only)')
else:
    from src.rag.qa_chain import ask_single_question, format_response
    
    question = 'What are the KYC requirements for opening a business checking account?'
    result = ask_single_question(question)
    print(format_response(result))

## 5. Customer Context Retrieval

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(os.environ['DATABASE_URL'])
customers = pd.read_sql('SELECT customer_id, name FROM customers LIMIT 5', engine)
print('Sample customers:')
print(customers)

In [ ]:
from src.rag.customer_context import get_full_customer_context, format_customer_context

customer_id = customers.iloc[0]['customer_id']
ctx = get_full_customer_context(customer_id)

print(f'Customer: {ctx["profile"]["name"]} ({customer_id})')
print(f'Trust Score: {ctx["trust_score"]["score"]}/100 ({ctx["trust_score"]["tier"]})')
print(f'Recent transactions: {len(ctx["last_5_transactions"])}')
print(f'Recommendations: {len(ctx.get("recommendations", []))}')
print()
print('Context string (first 500 chars):')
print(format_customer_context(ctx)[:500])

## 6. Collection Statistics

In [ ]:
from src.rag.vector_store import get_collection_stats

stats = get_collection_stats()
print(f'Total chunks indexed: {stats["total_chunks"]}')
print(f'\nDocuments in index:')
for doc in stats['documents']:
    print(f'  {doc["source"]:40s} {doc["chunk_count"]:4d} chunks  ({doc["doc_type"]})')